# Data Discovery in Data Lakes with BLEND

### Load libraries and define paths

In [1]:
import os
import sys
from pathlib import Path
import polars as pl
from tabulate import tabulate

In [2]:
sys.path.append(
    str(Path(os.path.abspath(os.path.curdir)).parent.absolute())
)


from blend import BLEND
from blend.indexing import index_tables

In [3]:
data_path = Path(os.path.abspath(os.path.curdir)).parent.joinpath("examples", "example-data", "modena")

data_lake_path = data_path.joinpath("data-lake")
index_db_path = data_path.joinpath("modena.db")
logdir_path = data_path.joinpath("log")
queries_path = data_path.joinpath("queries")

data_path.exists()

True

### Instantiate BLEND index

In [4]:
indexer = BLEND(index_db_path, clean_args={"lowercase": True})

In [5]:
load_opts = {"ignore_errors": True}
# index_tables(indexer, data_lake_path, True, None, 4, load_opts)

### Load the query dataset

We have some datasets in the _query_ folder:

In [6]:
queries = sorted(os.listdir(queries_path))

print('\n\n'.join(queries))

Archi-stradali.csv

Risultati-di-lista-delle-elezioni-europee-del-2024.cpy-0.csv

ds115_economia_spesa_media_mese_categoria_area_residenza_2007-2013.csv

section_district_codes.csv


In [7]:
# select one of the available queries
query_table_idx = 1
query_table_name = queries[query_table_idx]

# load the query dataset
qdf = pl.read_csv(queries_path.joinpath(query_table_name))

qdf

THE_PK_KEY,SECTION,DISTRICT,DISTRICT_CD,DATA_TYPE,ELECTION,YEAR,FORZA_ITALIA,SUDTIROLER_VOLKS_PARTEI,STATI_UNITI_D_EUROPA,ALTERNATIVA_POPOLARE___PPE,PACE_TERRA_DIGNITA_,MOVIMENTO_5_STELLE,LIBERTA_,ALLEANZA_VERDI_SINISTRA,PARTITO_DEMOCRATICO,FRATELLI_D_ITALIA,SIAMO_EUROPEI___AZIONE_CALENDA,LEGA,#REGISTERED,#VOTERS,#DISPUTED,#BLANK,#VALID,FORM,#NOT VALID,#NULL
str,i64,str,i64,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,i64,i64
"""1_Centro_Europee""",1,"""Centro""",1,"""Liste""","""Europee""",2024,30,null,28,8,11,20,null,48,136,186,null,23,881,533,0,6,525,"""E24""",8,2
"""2_Centro_Europee""",2,"""Centro""",1,"""Liste""","""Europee""",2024,29,3,11,1,13,null,null,33,null,95,22,null,732,444,0,3,431,"""E24""",13,10
null,5,"""Centro""",1,"""Liste""","""Europee""",2024,48,1,28,2,13,15,1,32,146,155,null,34,811,512,0,3,505,"""E24""",7,4
"""7_Centro_Europee""",7,"""Centro""",1,"""Liste""","""Europee""",2024,58,1,31,null,16,29,5,46,173,165,28,32,854,596,0,4,584,"""E24""",12,8
"""8_Centro_Europee""",8,"""Centro""",1,"""Liste""","""Europee""",2024,27,null,null,null,7,29,2,34,137,102,19,21,773,398,0,4,392,"""E24""",6,2
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
null,185,"""San Faustino""",null,"""Liste""","""Europee""",2024,21,1,12,0,8,null,1,56,266,153,32,21,845,618,0,5,609,"""E24""",9,4
"""187_Buon Pastore_Europee""",187,"""Buon Pastore""",3,"""Liste""","""Europee""",2024,11,0,9,1,3,31,1,35,204,93,13,19,663,426,0,3,420,"""E24""",6,3
"""188_Buon Pastore_Europee""",188,"""Buon Pastore""",3,"""Liste""","""Europee""",2024,21,null,19,2,18,34,4,36,239,82,18,29,718,521,0,5,502,"""E24""",19,14


## Keyword Search

In many use-cases, one of the simplest and most useful kind of data discovery task is the _keyword_ search.

Basically, we want to identify those datasets whose cell values, considered as a set, have the highest overlap with a user-given query set.

We don't check for any ordering on rows/columns, just the overlap.

In [8]:
# we flatten our query dataframe values to a set
# values = list(set(map(clean, {cell for row in qdf.rows() for cell in row})))
values = list({cell for row in qdf.rows() for cell in row})
len(values)

668

In [9]:
results = indexer.keyword_search(values, k=20)

print(f"Query table: {query_table_name}\n")

print(tabulate(results, headers=['dataset', 'overlap']))

Query table: Risultati-di-lista-delle-elezioni-europee-del-2024.cpy-0.csv

dataset                                                         overlap
------------------------------------------------------------  ---------
Risultati-di-lista-delle-elezioni-europee-del-2024                  667
Risultati-di-lista-delle-elezioni-europee-del-2024.cpy-0            667
Stradario-comunale                                                  536
Archi-stradali                                                      536
Sezioni-di-censimento                                               531
Pratiche-edilizie-anni-dal-1900-al-1999                             531
Pratiche-edilizie-anni-dal-2000-ad-oggi                             531
Servizio-di-scarico-risorsa-in-formato-CSV                          530
Risultati-delle-elezioni-europee-2019                               529
File-in-formato-csv                                                 519
Risultati-delle-elezioni-europee-anno-2014                   

## Unionable Table Search

In [14]:
table = qdf.rows()
results = indexer.union_search(table, 10)

print(f"Query table: {query_table_name}\n")
print(tabulate(results, headers=['dataset']))

Query table: Risultati-di-lista-delle-elezioni-europee-del-2024.cpy-0.csv

dataset
---------------------------------------------------------
File-in-formato-csv
Archi-stradali
Risultati-delle-elezioni-europee-2019
Risultati-di-lista-delle-elezioni-regionali-del-2024
Risultati-di-lista-delle-elezioni-europee-del-2024
Risultati-di-lista-delle-elezioni-amministrative-del-2009
Risultati-di-lista-delle-elezioni-amministrative-del-2014
Stradario-comunale
Risultati-di-lista-delle-elezioni-amministrative-del-2024
Risultati-di-lista-delle-elezioni-amministrative-del-2019


## Single Column JOIN Search

The dataset above has a single key column, _THE\_KEY_, which is the combination of _SECTION_, _DISTRICT\_CD_ and _ELECTION_

Such a combination might be useful to retrieve related tables using BLEND.

In [12]:
# extract and clean the values of the key
# column = qdf.get_column('THE_PK_KEY').map_elements(lambda x: clean(x), pl.String).drop_nulls()

column = qdf.get_column('THE_PK_KEY').drop_nulls().to_list()
column[:5]

['1_Centro_Europee',
 '2_Centro_Europee',
 '7_Centro_Europee',
 '8_Centro_Europee',
 '9_Centro_Europee']

Execute the search with BLEND, returning the 10 columns with highest overlap with the query. If we run the query several times with the same input, we 
should see always the same results (ties may appear in different order).

In [14]:
results = indexer.single_column_join_search(column, k=20)

print(f"Query table: {query_table_name}\n")
print(tabulate(results, headers=['dataset', 'column idx', 'overlap (distinct)', 'overlap (general)']))

Query table: Risultati-di-lista-delle-elezioni-europee-del-2024.cpy-0.csv

dataset                                                               column idx    overlap (distinct)
------------------------------------------------------------------  ------------  --------------------
Risultati-di-lista-delle-elezioni-europee-del-2024                             0                   128
Risultati-delle-elezioni-europee-2019                                          0                   128
Risultati-di-lista-delle-elezioni-europee-del-2024.cpy-0                       0                   128
Affluenze-delle-elezioni-europee-2019                                          0                   128
Affluenze-e-risultati-elettorali-delle-elezioni-europee-2024                   0                   128
Affluenze-delle-elezioni-europee-2009                                          0                   125
Risultati-delle-elezioni-europee-2009                                          0                   12

We can now easily identify the datasets we are most interested with; we can load them by accessing the results list and check
their content.

In [15]:
r_df = pl.read_csv(data_lake_path.joinpath(f"{results[-1][0]}.csv"))
r_df

THE_PK_KEY,SECTION,DISTRICT,DISTRICT_CD,DATA_TYPE,ELECTION,YEAR,ELECTION_CD,#MALE_REGISTERED,#FEMALE_REGISTERED,#MALE_VOTERS,#FEMALE_VOTERS
str,i64,str,i64,str,str,i64,str,i64,i64,i64,i64
"""160_San Faustino_Europee""",160,"""San Faustino""",4,"""Affluenza""","""Europee""",2019,"""EUR_2019""",394,418,296,319
"""161_San Faustino_Europee""",161,"""San Faustino""",4,"""Affluenza""","""Europee""",2019,"""EUR_2019""",281,309,188,198
"""164_San Faustino_Europee""",164,"""San Faustino""",null,"""Affluenza""","""Europee""",2019,"""EUR_2019""",383,421,279,303
"""166_San Faustino_Europee""",166,"""San Faustino""",4,"""Affluenza""","""Europee""",2019,"""EUR_2019""",357,327,237,229
null,167,"""San Faustino""",4,"""Affluenza""","""Europee""",2019,null,395,434,296,325
…,…,…,…,…,…,…,…,…,…,…,…
"""154_San Faustino_Europee""",154,"""San Faustino""",4,"""Affluenza""","""Europee""",2019,"""EUR_2019""",340,383,249,280
"""156_San Faustino_Europee""",156,"""San Faustino""",4,"""Affluenza""","""Europee""",2019,null,373,394,262,273
"""157_San Faustino_Europee""",157,"""San Faustino""",4,"""Affluenza""","""Europee""",2019,"""EUR_2019""",359,374,256,253


## Multi-Column JOIN Search - Combination of single-JOIN searches

In many cases, a single column doesn't identify every record of a dataset, and a combination of different attributes is thus required.

Suppose that we do not have anymore the "THE_KEY" column.

In [14]:
qdf = qdf.drop("THE_PK_KEY")
qdf.head()

SECTION,DISTRICT,DISTRICT_CD,DATA_TYPE,ELECTION,YEAR,FORZA_ITALIA,SUDTIROLER_VOLKS_PARTEI,STATI_UNITI_D_EUROPA,ALTERNATIVA_POPOLARE___PPE,PACE_TERRA_DIGNITA_,MOVIMENTO_5_STELLE,LIBERTA_,ALLEANZA_VERDI_SINISTRA,PARTITO_DEMOCRATICO,FRATELLI_D_ITALIA,SIAMO_EUROPEI___AZIONE_CALENDA,LEGA,#REGISTERED,#VOTERS,#DISPUTED,#BLANK,#VALID,FORM,#NOT VALID,#NULL
i64,str,i64,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,i64,i64
1,"""Centro""",1,"""Liste""","""Europee""",2024,30,null,28,8,11,20,null,48,136,186,null,23,881,533,0,6,525,"""E24""",8,2
2,"""Centro""",1,"""Liste""","""Europee""",2024,29,3,11,1,13,null,null,33,null,95,22,null,732,444,0,3,431,"""E24""",13,10
5,"""Centro""",1,"""Liste""","""Europee""",2024,48,1,28,2,13,15,1,32,146,155,null,34,811,512,0,3,505,"""E24""",7,4
7,"""Centro""",1,"""Liste""","""Europee""",2024,58,1,31,null,16,29,5,46,173,165,28,32,854,596,0,4,584,"""E24""",12,8
8,"""Centro""",1,"""Liste""","""Europee""",2024,27,null,null,null,7,29,2,34,137,102,19,21,773,398,0,4,392,"""E24""",6,2


How can we search joinable tables on "SECTION", "DISTRICT_CD", and "ELECTION" columns at the same time?

To identify joinable tables on multiple columns, can we run several single-column searches? Is this a good option?

In [15]:
column = qdf.get_column('SECTION').drop_nulls().to_list()
results_section = indexer.single_column_join_search(column, k=10)

In [16]:
column = qdf.get_column('DISTRICT').drop_nulls().to_list()
results_district = indexer.single_column_join_search(column, k=10)

In [17]:
column = qdf.get_column('ELECTION').drop_nulls().to_list()
results_election = indexer.single_column_join_search(column, k=10)

In [18]:
from collections import defaultdict


aggregation = defaultdict(int)

for results in [results_section, results_district, results_election]:
    for table, _, _ in results:
        aggregation[table] += 1

In [19]:
results = list(sorted(list(aggregation.items()), key=lambda r: r[1], reverse=True))

# select the top-k
results = results[:10]

print(tabulate(results, ['dataset', 'occurrences']))

dataset                                                                      occurrences
-------------------------------------------------------------------------  -------------
Affluenze-e-risultati-elettorali-delle-elezioni-europee-2024                           3
Risultati-di-lista-delle-elezioni-amministrative-del-2024                              2
Affluenze-e-risultati-elettorali-delle-elezioni-amministrative-2024.cpy-0              1
Pratiche-edilizie-anni-dal-2000-ad-oggi                                                1
Risultati-di-lista-delle-elezioni-regionali-del-2024.unpivot.cpy-0                     1
Affluenze-delle-elezioni-amministrative-2019                                           1
Risultati-di-lista-delle-elezioni-europee-del-2024.cpy-0                               1
Risultati-delle-elezioni-europee-2019.unpivot                                          1
File-in-formato-csv                                                                    1
Risultati-di-lista-de

By combining results from different single-column searches we have some drawbacks in the end: 

- the order isn't always the same, 
- can be costly, if we need to run it on a high number of different columns,
- the alignment of the rows isn't guaranteed.

## Multi-Column JOIN Search - MATE algorithm

Instead, we can use a **multi-column search** approach. 

This is based on MATE (Multi-Attribute Table Extraction) algorithm, which allows us to search n-ary joins without any
other intermediate step.

In [8]:
table = qdf.select(['SECTION', 'DISTRICT', 'ELECTION']).rows()

print(tabulate(table[:10]))

--  ------  -------
 1  Centro  Europee
 2  Centro  Europee
 5  Centro  Europee
 7  Centro  Europee
 8  Centro  Europee
 9  Centro  Europee
10  Centro  Europee
11  Centro  Europee
12  Centro  Europee
14  Centro  Europee
--  ------  -------


In [ ]:
mc_results = indexer.multi_column_join_search(table, 10, verbose=True)

Evaluating candidate table rows: 100%|█████████████████████████████████████████| 28535/28535 [00:00<00:00, 97179.06it/s]


In [17]:
print(f"Query table: {query_table_name}\n")
print(tabulate(mc_results, headers=['dataset', 'columns', 'join_score']))

Query table: Risultati-di-lista-delle-elezioni-europee-del-2024.cpy-0.csv

dataset                                                           columns      join_score
----------------------------------------------------------------  ---------  ------------
Risultati-delle-elezioni-europee-2019.unpivot.cpy-0               [0, 1, 3]          2423
Risultati-delle-elezioni-europee-2019.unpivot.cpy-1               [0, 1, 3]          2190
Risultati-delle-elezioni-europee-2019.unpivot                     [0, 1, 3]          2736
Risultati-delle-elezioni-europee-2009.unpivot                     [0, 1, 3]          2235
Risultati-delle-elezioni-europee-anno-2014.unpivot                [0, 1, 3]          2235
Risultati-delle-elezioni-europee-anno-2014.unpivot.cpy-0          [0, 1, 3]          1795
Risultati-di-lista-delle-elezioni-europee-del-2024.unpivot        [0, 1, 3]          1976
Risultati-delle-elezioni-europee-anno-2014.unpivot.cpy-1          [0, 1, 3]          1795
Risultati-delle-elezioni-

The order of the columns **doesn't affect** the final results, but might impact the efficiency (see section 6.1 of MATE paper if you are interested).

The final order in the top-K might slightly change, but overall the top-K tables should be the same.

We can swap the columns used before:

In [18]:
table = qdf.select(['DISTRICT', 'SECTION', 'ELECTION']).rows()
print(tabulate(table[:5]))

------  -  -------
Centro  1  Europee
Centro  2  Europee
Centro  5  Europee
Centro  7  Europee
Centro  8  Europee
------  -  -------


In [19]:
mc_results_v2 = indexer.multi_column_join_search(table, 10, verbose=True)

Evaluating candidate table rows: 100%|██████████████████████████████████████| 799177/799177 [00:03<00:00, 232971.70it/s]


In [20]:
# the results are the same obtained above
print(tabulate(mc_results_v2, headers=['dataset', 'columns', 'join_score']))

dataset                                                           columns      join_score
----------------------------------------------------------------  ---------  ------------
Risultati-delle-elezioni-europee-2019.unpivot.cpy-0               [1, 0, 3]          3505
Risultati-di-lista-delle-elezioni-europee-del-2024.unpivot        [1, 0, 3]          3249
Risultati-delle-elezioni-europee-2019.unpivot                     [1, 0, 3]          4248
Risultati-delle-elezioni-europee-2009.unpivot                     [1, 0, 3]          3545
Risultati-delle-elezioni-europee-anno-2014.unpivot                [1, 0, 3]          3545
Risultati-delle-elezioni-europee-2009.unpivot.cpy-1               [1, 0, 3]          2652
Risultati-delle-elezioni-europee-2019.unpivot.cpy-1               [1, 0, 3]          3153
Risultati-delle-elezioni-europee-anno-2014.unpivot.cpy-1          [1, 0, 3]          2631
Risultati-delle-elezioni-europee-anno-2014.unpivot.cpy-0          [1, 0, 3]          2628
Risultati-

In [21]:
tables_from_run_1 = {r[0] for r in mc_results}
tables_from_run_2 = {r[0] for r in mc_results_v2}

len(tables_from_run_1.intersection(tables_from_run_2)), tables_from_run_1.difference(tables_from_run_2), tables_from_run_2.difference(tables_from_run_1)

(10, set(), set())

We can do another test with a different combination of the same three columns:

In [22]:
qdf.get_column('ELECTION').unique()

ELECTION
str
"""Europee"""


In [23]:
table = qdf.select(['ELECTION', 'DISTRICT', 'SECTION']).rows()
print(tabulate(table[:3]))

-------  ------  -
Europee  Centro  1
Europee  Centro  2
Europee  Centro  5
-------  ------  -


In [26]:
mc_results_v3 = indexer.multi_column_join_search(table, 10, verbose=True)

Evaluating candidate table rows: 100%|███████████████████████████████████████| 799238/799238 [00:09<00:00, 81812.11it/s]


In [27]:
tables_from_run_1 = {r[0] for r in mc_results}
tables_from_run_2 = {r[0] for r in mc_results_v2}
tables_from_run_3 = {r[0] for r in mc_results_v3}

len(tables_from_run_1.intersection(tables_from_run_2).intersection(tables_from_run_3))

10